# Assignment 2: Spark DataFrames and Transformations

## a) Theory

### Question:
Differentiate between transformations and actions in Spark. Provide at least 3 examples for each and explain their significance in execution planning.

### Answer:

**Transformations:**
- Lazy operations that create a new RDD/DataFrame from an existing one
- Do not execute immediately; only record the operation in a DAG
- Return a new DataFrame/RDD without modifying the original
- Executed only when an action is called

**Examples of Transformations:**
1. **`select()`** - Selects specific columns from DataFrame
2. **`filter()`/`where()`** - Filters rows based on conditions
3. **`groupBy()`** - Groups data based on column values
4. **`orderBy()`/`sort()`** - Sorts data
5. **`join()`** - Joins two DataFrames

**Actions:**
- Trigger the execution of all recorded transformations
- Return values to the driver program or write data to external storage
- Cause Spark to execute the DAG of transformations
- Produce actual output/results

**Examples of Actions:**
1. **`show()`** - Displays rows of DataFrame to console
2. **`count()`** - Returns the number of rows
3. **`collect()`** - Returns all rows as an array to driver
4. **`take(n)`** - Returns first n rows
5. **`write()`** - Writes data to external storage

**Significance in Execution Planning:**
- **Lazy Evaluation**: Transformations allow Spark to optimize the entire execution plan before running
- **DAG Optimization**: Spark builds a Directed Acyclic Graph and optimizes it (combining operations, pipelining)
- **Efficiency**: Unnecessary transformations can be eliminated
- **Fault Tolerance**: Transformation lineage allows recomputation of lost partitions
- **Actions trigger execution**: Only when action is called, the optimized plan executes across the cluster

## b) Practical / Coding

### Question:
Using PySpark:
- Load a sample dataset (CSV or manually created)
- Perform the following transformations:
  - Select specific columns
  - Apply a filter condition
  - Group data based on a column and calculate count
- Display the results using appropriate actions

### Solution:

In [0]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, sum
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, FloatType

In [0]:
# Create Spark Session
spark = SparkSession.builder.getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


### Method 1: Load Manually Created Dataset

In [0]:
# Define schema for employee dataset
schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Department", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Salary", FloatType(), True),
    StructField("Experience", IntegerType(), True)
])

In [0]:
# Create sample employee data
data = [
    (101, "Alice Johnson", "IT", 28, 55000.0, 3),
    (102, "Bob Smith", "HR", 35, 48000.0, 8),
    (103, "Charlie Brown", "IT", 30, 62000.0, 5),
    (104, "Diana Prince", "Finance", 32, 58000.0, 6),
    (105, "Eve Davis", "IT", 26, 51000.0, 2),
    (106, "Frank Miller", "HR", 40, 45000.0, 12),
    (107, "Grace Lee", "Finance", 29, 60000.0, 4),
    (108, "Henry Wilson", "IT", 33, 67000.0, 7),
    (109, "Ivy Martinez", "Finance", 31, 56000.0, 5),
    (110, "Jack Taylor", "HR", 27, 42000.0, 3)
]

In [0]:
# Create DataFrame from sample data
df = spark.createDataFrame(data, schema)
print("DataFrame created successfully!\n")

# Display the original dataset
print("=== Original Dataset ===")
df.show()

DataFrame created successfully!

=== Original Dataset ===
+----------+-------------+----------+---+-------+----------+
|EmployeeID|         Name|Department|Age| Salary|Experience|
+----------+-------------+----------+---+-------+----------+
|       101|Alice Johnson|        IT| 28|55000.0|         3|
|       102|    Bob Smith|        HR| 35|48000.0|         8|
|       103|Charlie Brown|        IT| 30|62000.0|         5|
|       104| Diana Prince|   Finance| 32|58000.0|         6|
|       105|    Eve Davis|        IT| 26|51000.0|         2|
|       106| Frank Miller|        HR| 40|45000.0|        12|
|       107|    Grace Lee|   Finance| 29|60000.0|         4|
|       108| Henry Wilson|        IT| 33|67000.0|         7|
|       109| Ivy Martinez|   Finance| 31|56000.0|         5|
|       110|  Jack Taylor|        HR| 27|42000.0|         3|
+----------+-------------+----------+---+-------+----------+



In [0]:
# Display schema of the DataFrame
print("=== DataFrame Schema ===")
df.printSchema()

=== DataFrame Schema ===
root
 |-- EmployeeID: integer (nullable = true)
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Salary: float (nullable = true)
 |-- Experience: integer (nullable = true)



### Transformation 1: Select Specific Columns

In [0]:
# Select specific columns: Name, Department, and Salary
# This is a TRANSFORMATION (lazy - not executed yet)
selected_df = df.select("Name", "Department", "Salary")

print("=== Selected Columns: Name, Department, Salary ===")
# show() is an ACTION - triggers execution
selected_df.show()

=== Selected Columns: Name, Department, Salary ===
+-------------+----------+-------+
|         Name|Department| Salary|
+-------------+----------+-------+
|Alice Johnson|        IT|55000.0|
|    Bob Smith|        HR|48000.0|
|Charlie Brown|        IT|62000.0|
| Diana Prince|   Finance|58000.0|
|    Eve Davis|        IT|51000.0|
| Frank Miller|        HR|45000.0|
|    Grace Lee|   Finance|60000.0|
| Henry Wilson|        IT|67000.0|
| Ivy Martinez|   Finance|56000.0|
|  Jack Taylor|        HR|42000.0|
+-------------+----------+-------+



In [0]:
# Alternative method using col() function
selected_df_alt = df.select(col("EmployeeID"), col("Name"), col("Age"))

print("=== Selected Columns: EmployeeID, Name, Age ===")
selected_df_alt.show()

=== Selected Columns: EmployeeID, Name, Age ===
+----------+-------------+---+
|EmployeeID|         Name|Age|
+----------+-------------+---+
|       101|Alice Johnson| 28|
|       102|    Bob Smith| 35|
|       103|Charlie Brown| 30|
|       104| Diana Prince| 32|
|       105|    Eve Davis| 26|
|       106| Frank Miller| 40|
|       107|    Grace Lee| 29|
|       108| Henry Wilson| 33|
|       109| Ivy Martinez| 31|
|       110|  Jack Taylor| 27|
+----------+-------------+---+



### Transformation 2: Apply Filter Condition

In [0]:
# Filter employees with salary greater than 50000
# This is a TRANSFORMATION (lazy)
filtered_df = df.filter(col("Salary") > 50000)

print("=== Employees with Salary > 50000 ===")
filtered_df.show()

=== Employees with Salary > 50000 ===
+----------+-------------+----------+---+-------+----------+
|EmployeeID|         Name|Department|Age| Salary|Experience|
+----------+-------------+----------+---+-------+----------+
|       101|Alice Johnson|        IT| 28|55000.0|         3|
|       103|Charlie Brown|        IT| 30|62000.0|         5|
|       104| Diana Prince|   Finance| 32|58000.0|         6|
|       105|    Eve Davis|        IT| 26|51000.0|         2|
|       107|    Grace Lee|   Finance| 29|60000.0|         4|
|       108| Henry Wilson|        IT| 33|67000.0|         7|
|       109| Ivy Martinez|   Finance| 31|56000.0|         5|
+----------+-------------+----------+---+-------+----------+



In [0]:
# Multiple filter conditions: IT department with salary > 55000
multi_filtered_df = df.filter((col("Department") == "IT") & (col("Salary") > 55000))

print("=== IT Department Employees with Salary > 55000 ===")
multi_filtered_df.show()

=== IT Department Employees with Salary > 55000 ===
+----------+-------------+----------+---+-------+----------+
|EmployeeID|         Name|Department|Age| Salary|Experience|
+----------+-------------+----------+---+-------+----------+
|       103|Charlie Brown|        IT| 30|62000.0|         5|
|       108| Henry Wilson|        IT| 33|67000.0|         7|
+----------+-------------+----------+---+-------+----------+



In [0]:
# Filter using SQL-style string expression
sql_filtered_df = df.filter("Age >= 30 AND Experience >= 5")

print("=== Employees with Age >= 30 and Experience >= 5 ===")
sql_filtered_df.show()

=== Employees with Age >= 30 and Experience >= 5 ===
+----------+-------------+----------+---+-------+----------+
|EmployeeID|         Name|Department|Age| Salary|Experience|
+----------+-------------+----------+---+-------+----------+
|       102|    Bob Smith|        HR| 35|48000.0|         8|
|       103|Charlie Brown|        IT| 30|62000.0|         5|
|       104| Diana Prince|   Finance| 32|58000.0|         6|
|       106| Frank Miller|        HR| 40|45000.0|        12|
|       108| Henry Wilson|        IT| 33|67000.0|         7|
|       109| Ivy Martinez|   Finance| 31|56000.0|         5|
+----------+-------------+----------+---+-------+----------+



### Transformation 3: Group Data and Calculate Count

In [0]:
# Group by Department and count employees
# groupBy() is a TRANSFORMATION, count() here is an aggregation function
grouped_df = df.groupBy("Department").count()

print("=== Employee Count by Department ===")
# show() is the ACTION that triggers execution
grouped_df.show()

=== Employee Count by Department ===
+----------+-----+
|Department|count|
+----------+-----+
|        IT|    4|
|        HR|    3|
|   Finance|    3|
+----------+-----+



In [0]:
# Group by Department with multiple aggregations
agg_df = df.groupBy("Department").agg(
    count("EmployeeID").alias("Total_Employees"),
    avg("Salary").alias("Average_Salary"),
    sum("Salary").alias("Total_Salary")
)

print("=== Department-wise Statistics ===")
agg_df.show()

=== Department-wise Statistics ===
+----------+---------------+--------------+------------+
|Department|Total_Employees|Average_Salary|Total_Salary|
+----------+---------------+--------------+------------+
|        IT|              4|       58750.0|    235000.0|
|        HR|              3|       45000.0|    135000.0|
|   Finance|              3|       58000.0|    174000.0|
+----------+---------------+--------------+------------+



In [0]:
# Sort the grouped data by count in descending order
sorted_grouped_df = grouped_df.orderBy(col("count").desc())

print("=== Department Employee Count (Sorted) ===")
sorted_grouped_df.show()

=== Department Employee Count (Sorted) ===
+----------+-----+
|Department|count|
+----------+-----+
|        IT|    4|
|        HR|    3|
|   Finance|    3|
+----------+-----+



### Combining Multiple Transformations and Actions

In [0]:
# Chain multiple transformations: Select -> Filter -> Group -> Sort
result_df = df.select("Name", "Department", "Salary", "Age") \
    .filter(col("Salary") > 45000) \
    .groupBy("Department") \
    .agg(count("Name").alias("HighEarners"), 
         avg("Salary").alias("AvgSalary")) \
    .orderBy(col("HighEarners").desc())

print("=== High Earners (Salary > 45000) by Department ===")
result_df.show()

=== High Earners (Salary > 45000) by Department ===
+----------+-----------+---------+
|Department|HighEarners|AvgSalary|
+----------+-----------+---------+
|        IT|          4|  58750.0|
|   Finance|          3|  58000.0|
|        HR|          1|  48000.0|
+----------+-----------+---------+



### Demonstrating Different Actions

In [0]:
# ACTION 1: count() - returns number of rows
total_employees = df.count()
print(f"Total number of employees: {total_employees}")

Total number of employees: 10


In [0]:
# ACTION 2: collect() - returns all rows as list
# Use with caution on large datasets
it_employees = df.filter(col("Department") == "IT").collect()
print(f"\nIT Department Employees (using collect()):")
for emp in it_employees:
    print(f"  {emp.Name} - ${emp.Salary}")


IT Department Employees (using collect()):
  Alice Johnson - $55000.0
  Charlie Brown - $62000.0
  Eve Davis - $51000.0
  Henry Wilson - $67000.0


In [0]:
# ACTION 3: take() - returns first n rows
top_3 = df.orderBy(col("Salary").desc()).take(3)
print("\nTop 3 Highest Paid Employees:")
for i, emp in enumerate(top_3, 1):
    print(f"  {i}. {emp.Name} - ${emp.Salary}")


Top 3 Highest Paid Employees:
  1. Henry Wilson - $67000.0
  2. Charlie Brown - $62000.0
  3. Grace Lee - $60000.0


In [0]:
# ACTION 4: first() - returns first row
first_record = df.first()
print(f"\nFirst Employee Record: {first_record.Name}, {first_record.Department}")


First Employee Record: Alice Johnson, IT


In [0]:
# ACTION 5: describe() - statistical summary
print("\n=== Statistical Summary ===")
df.describe(["Age", "Salary", "Experience"]).show()


=== Statistical Summary ===
+-------+-----------------+-----------------+------------------+
|summary|              Age|           Salary|        Experience|
+-------+-----------------+-----------------+------------------+
|  count|               10|               10|                10|
|   mean|             31.1|          54400.0|               5.5|
| stddev|4.175324338699131|7876.829593462363|2.9533408577782247|
|    min|               26|          42000.0|                 2|
|    max|               40|          67000.0|                12|
+-------+-----------------+-----------------+------------------+



### Method 2: Creating and Reading from CSV (Optional)

In [0]:
from pyspark.sql import SparkSession
# Create a new Spark session
spark = SparkSession.builder.appName('MyApp').getOrCreate()

# Create schema and volume once

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.default")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.streaming_volume")

# Save DataFrame as CSV to the Unity Catalog volume instead of /tmp
output_path = "/Volumes/workspace/default/streaming_volume/employees"

df.write.mode("overwrite").option("header", "true").csv(output_path)

print(f"Saved to: {output_path}")

Saved to: /Volumes/workspace/default/streaming_volume/employees


In [0]:
# Read data from CSV file
df_from_csv = spark.read.option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/streaming_volume/employees")

df_from_csv.show(5)

+----------+------------+----------+---+-------+----------+
|EmployeeID|        Name|Department|Age| Salary|Experience|
+----------+------------+----------+---+-------+----------+
|       109|Ivy Martinez|   Finance| 31|56000.0|         5|
|       110| Jack Taylor|        HR| 27|42000.0|         3|
|       104|Diana Prince|   Finance| 32|58000.0|         6|
|       105|   Eve Davis|        IT| 26|51000.0|         2|
|       107|   Grace Lee|   Finance| 29|60000.0|         4|
+----------+------------+----------+---+-------+----------+
only showing top 5 rows


In [0]:
# Stop Spark Session
spark.stop()
print("\nSpark Session stopped successfully!")


Spark Session stopped successfully!


### Summary of Transformations vs Actions Used

## TRANSFORMATIONS USED (Lazy Operations)
- select() — Selected specific columns  
- filter() / where() — Applied filter conditions  
- groupBy() — Grouped data by department  
- orderBy() / sort() — Sorted results  
- agg() — Applied aggregation functions  

## ACTIONS USED (Trigger Execution)
- show() — Display data in tabular format  
- count() — Count number of rows  
- collect() — Retrieve all rows to driver  
- take(n) — Retrieve first n rows  
- first() — Retrieve first row  
- describe() — Statistical summary  
- write() — Save data to storage  

In [0]:
# Stop Spark Session
spark.stop()
print("\nSpark Session stopped successfully!")


Spark Session stopped successfully!
